# K-Means Clustering Exploration (A1, A2, A3)

This notebook covers:
- **A1**: K-Means MCQs concepts (objective, K parameter, steps, distance metrics, centroids, elbow method)
- **A2**: Unsupervised learning concepts (clustering vs classification, SSE, feature scaling, outliers)
- **A3**: Numerical problems (manual iterations, centroid updates, SSE calculation)

## Setup

In [ ]:
# Setup imports - Self-contained notebook (no external package imports)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans as SklearnKMeans

# Configure matplotlib for inline display
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Imports successful - Self-contained notebook")

## 1. Generate Sample Data

Create synthetic data with 3 clusters for demonstration.

In [ ]:
# Generate sample data
def generate_sample_data(n_samples=300, centers=3, n_features=2, cluster_std=1.5, random_state=42):
    """Generate sample data for clustering demonstration."""
    X, y = make_blobs(n_samples=n_samples, centers=centers, n_features=n_features, 
                     cluster_std=cluster_std, random_state=random_state)
    # Add some noise
    X += np.random.normal(0, 0.2, X.shape)
    return X, y

X, true_labels = generate_sample_data()

print(f"Data shape: {X.shape}")
print(f"True labels shape: {true_labels.shape}")
print(f"Number of clusters: {len(np.unique(true_labels))}")

# Visualize the raw data
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, s=50)
plt.title('Raw Data (Before Clustering)', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

## 2. K-Means Implementation from Scratch

Define the KMeansFromScratch class directly in this notebook (self-contained).

In [ ]:
# K-Means from Scratch Implementation
class KMeansFromScratch:
    """K-Means clustering algorithm implemented from scratch."""
    
    def __init__(self, n_clusters=3, max_iter=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia = None
        
    def fit(self, X):
        """Fit K-Means clustering to the data."""
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        n_samples = X.shape[0]
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices].copy()
        
        for iteration in range(self.max_iter):
            # Step 1: Assign each point to nearest centroid
            distances = self._compute_distances(X)
            new_labels = np.argmin(distances, axis=1)
            
            # Step 2: Update centroids
            new_centroids = self._update_centroids(X, new_labels)
            
            # Check for convergence
            if np.allclose(self.centroids, new_centroids):
                break
                
            self.centroids = new_centroids
            self.labels = new_labels
            
        # Calculate inertia (within-cluster sum of squared distances)
        self.inertia = self._compute_inertia(X)
        return self
        
    def _compute_distances(self, X):
        """Compute distances from each point to each centroid."""
        n_samples = X.shape[0]
        distances = np.zeros((n_samples, self.n_clusters))
        
        for i in range(self.n_clusters):
            distances[:, i] = np.linalg.norm(X - self.centroids[i], axis=1)
            
        return distances
    
    def _update_centroids(self, X, labels):
        """Update centroids based on current cluster assignments."""
        new_centroids = np.zeros_like(self.centroids)
        
        for cluster in range(self.n_clusters):
            cluster_points = X[labels == cluster]
            if len(cluster_points) > 0:
                new_centroids[cluster] = np.mean(cluster_points, axis=0)
            else:
                # Reinitialize empty cluster
                random_idx = np.random.randint(0, X.shape[0])
                new_centroids[cluster] = X[random_idx]
                
        return new_centroids
    
    def _compute_inertia(self, X):
        """Calculate within-cluster sum of squared distances."""
        inertia = 0.0
        for cluster in range(self.n_clusters):
            cluster_points = X[self.labels == cluster]
            if len(cluster_points) > 0:
                distances = np.linalg.norm(cluster_points - self.centroids[cluster], axis=1)
                inertia += np.sum(distances ** 2)
        return inertia
    
    def predict(self, X):
        """Predict cluster labels for new data."""
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)
    
    def fit_predict(self, X):
        """Fit the model and return cluster labels."""
        self.fit(X)
        return self.labels

print("✅ KMeansFromScratch class defined")

## 3. K-Means with Different K Values

Apply K-Means from scratch with K=2, 3, 4 and visualize results.

In [ ]:
# K-Means with different K values
def visualize_kmeans(X, labels, centroids, title, ax=None):
    """Visualize K-Means clustering results."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.6, s=50)
    ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, linewidths=3, label='Centroids')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend()
    ax.grid(True, alpha=0.3)
    return scatter

# Test with different K values
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, k in enumerate([2, 3, 4]):
    kmeans = KMeansFromScratch(n_clusters=k, random_state=42)
    predicted_labels = kmeans.fit_predict(X)
    visualize_kmeans(X, predicted_labels, kmeans.centroids, f'K-Means (K={k})', ax=axes[idx])
    print(f"K={k}: Inertia = {kmeans.inertia:.2f}")

plt.tight_layout()
plt.show()

## 3. Elbow Method for Optimal K

Apply the elbow method to determine the optimal number of clusters.

In [ ]:
# Elbow method
def elbow_method(X, max_k=10, random_state=42):
    """Apply elbow method to determine optimal number of clusters."""
    inertias = []
    k_range = range(1, max_k + 1)
    
    for k in k_range:
        kmeans = KMeansFromScratch(n_clusters=k, random_state=random_state)
        kmeans.fit(X)
        inertias.append(kmeans.inertia)
    
    return k_range, inertias

k_range, inertias = elbow_method(X, max_k=10)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method for Optimal K', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squared Distances)')
plt.grid(True, alpha=0.3)

# Annotate the elbow point
elbow_k = 3
elbow_inertia = inertias[elbow_k - 1]
plt.annotate(f'Elbow at K={elbow_k}', 
             xy=(elbow_k, elbow_inertia), 
             xytext=(elbow_k + 1, elbow_inertia + 2000),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=12, color='red')

plt.show()

print("Inertia values:")
for k, inertia in zip(k_range, inertias):
    print(f"  K={k}: {inertia:.2f}")

## 4. A3 Numerical Problems - Step-by-Step Solutions

Solve the 6 numerical problems from A3 Problems K means.docx

In [ ]:
# Problem 1: One-dimensional data points
# X = {2, 4, 6, 8, 10}, initial centroids: C1 = 3, C2 = 9

print("=" * 60)
print("PROBLEM 1: One-dimensional K-Means")
print("=" * 60)

X_1d = np.array([2, 4, 6, 8, 10]).reshape(-1, 1)
centroids = np.array([3, 9]).reshape(-1, 1)

print(f"Data points: {X_1d.flatten()}")
print(f"Initial centroids: {centroids.flatten()}")

# Step 1: Calculate distances
distances = np.abs(X_1d - centroids.T)  # Shape: (5, 2)
print(f"\nDistances to centroids:")
for i, point in enumerate(X_1d.flatten()):
    print(f"  Point {point}: d(C1)={distances[i,0]:.1f}, d(C2)={distances[i,1]:.1f}")

# Step 2: Assign to nearest centroid
assignments = np.argmin(distances, axis=1)
print(f"\nCluster assignments: {assignments}")
print(f"Cluster 0: {X_1d[assignments==0].flatten()}")
print(f"Cluster 1: {X_1d[assignments==1].flatten()}")

# Step 3: Calculate new centroids
new_centroids = np.array([X_1d[assignments==0].mean(), X_1d[assignments==1].mean()])
print(f"\nNew centroids: {new_centroids.flatten()}")

# Step 4: Final assignments after first iteration
print(f"\nFinal cluster assignments after 1st iteration:")
print(f"  Cluster 1: {X_1d[assignments==0].flatten()}")
print(f"  Cluster 2: {X_1d[assignments==1].flatten()}")

In [ ]:
# Problem 2: Six 2D points
print("=" * 60)
print("PROBLEM 2: Six 2D Points K-Means")
print("=" * 60)

points = {
    'A': np.array([1, 1]),
    'B': np.array([2, 2]),
    'C': np.array([3, 1]),
    'D': np.array([8, 8]),
    'E': np.array([9, 7]),
    'F': np.array([8, 9])
}

X_2d = np.array(list(points.values()))
labels_list = list(points.keys())
initial_centroids = np.array([[1, 1], [8, 8]])

print(f"Points:")
for name, pt in points.items():
    print(f"  {name}: {pt}")
print(f"Initial centroids: C1={initial_centroids[0]}, C2={initial_centroids[1]}")

# Calculate distances
def euclidean_dist(p1, p2):
    return np.linalg.norm(p1 - p2)

print("\nDistances:")
for i, (name, pt) in enumerate(points.items()):
    d1 = euclidean_dist(pt, initial_centroids[0])
    d2 = euclidean_dist(pt, initial_centroids[1])
    cluster = 1 if d1 <= d2 else 2
    print(f"  {name}{pt}: d(C1)={d1:.2f}, d(C2)={d2:.2f} → Cluster {cluster}")

# Assignments
assignments = []
for pt in X_2d:
    d1 = euclidean_dist(pt, initial_centroids[0])
    d2 = euclidean_dist(pt, initial_centroids[1])
    assignments.append(0 if d1 <= d2 else 1)
assignments = np.array(assignments)

print(f"\nCluster assignments:")
cluster_1 = X_2d[assignments==0]
cluster_2 = X_2d[assignments==1]
print(f"  Cluster 1: {cluster_1}")
print(f"  Cluster 2: {cluster_2}")

# New centroids
new_c1 = cluster_1.mean(axis=0)
new_c2 = cluster_2.mean(axis=0)
print(f"\nNew centroids:")
print(f"  C1_new = {new_c1}")
print(f"  C2_new = {new_c2}")

In [ ]:
# Problem 3: SSE Calculation
print("=" * 60)
print("PROBLEM 3: SSE Calculation")
print("=" * 60)

# Same data as Problem 2
X_2d = np.array([[1, 1], [2, 2], [3, 1], [8, 8], [9, 7], [8, 9]])
initial_centroids = np.array([[1, 1], [8, 8]])

# After one iteration (from Problem 2)
assignments = np.array([0, 0, 0, 1, 1, 1])
new_centroids = np.array([[2.0, 1.333], [8.333, 8.0]])

print(f"Cluster assignments: {assignments}")
print(f"New centroids: C1={new_centroids[0]}, C2={new_centroids[1]}")

# Calculate SSE
def calculate_sse(X, assignments, centroids):
    sse = 0.0
    for cluster in range(len(centroids)):
        cluster_points = X[assignments == cluster]
        if len(cluster_points) > 0:
            distances = np.linalg.norm(cluster_points - centroids[cluster], axis=1)
            sse += np.sum(distances ** 2)
    return sse

sse = calculate_sse(X_2d, assignments, new_centroids)
print(f"\nWithin-cluster SSE: {sse:.4f}")

# Detailed calculation
print("\nDetailed SSE calculation:")
for cluster in range(2):
    cluster_points = X_2d[assignments == cluster]
    centroid = new_centroids[cluster]
    cluster_sse = 0
    for pt in cluster_points:
        dist_sq = np.sum((pt - centroid) ** 2)
        cluster_sse += dist_sq
        print(f"  Point {pt} to centroid {centroid}: squared distance = {dist_sq:.4f}")
    print(f"  Cluster {cluster+1} SSE: {cluster_sse:.4f}")

In [ ]:
# Problem 4: Four points with K=2
print("=" * 60)
print("PROBLEM 4: Four Points K-Means")
print("=" * 60)

X_4 = np.array([[1, 1], [2, 1], [4, 3], [5, 4]])
initial_centroids = np.array([[1, 1], [5, 4]])

print(f"Points: {X_4}")
print(f"Initial centroids: C1={initial_centroids[0]}, C2={initial_centroids[1]}")

# One iteration
distances = np.zeros((4, 2))
for i in range(4):
    distances[i, 0] = np.linalg.norm(X_4[i] - initial_centroids[0])
    distances[i, 1] = np.linalg.norm(X_4[i] - initial_centroids[1])

assignments = np.argmin(distances, axis=1)
print(f"\nDistances and assignments:")
for i in range(4):
    print(f"  Point {X_4[i]}: d(C1)={distances[i,0]:.2f}, d(C2)={distances[i,1]:.2f} → Cluster {assignments[i]+1}")

# New centroids
new_c1 = X_4[assignments==0].mean(axis=0)
new_c2 = X_4[assignments==1].mean(axis=0)
print(f"\nNew centroids: C1={new_c1}, C2={new_c2}")

# SSE
sse = calculate_sse(X_4, assignments, np.array([new_c1, new_c2]))
print(f"Total within-cluster SSE: {sse:.4f}")

# Check if another iteration changes assignments
print("\nChecking convergence (2nd iteration):")
distances2 = np.zeros((4, 2))
for i in range(4):
    distances2[i, 0] = np.linalg.norm(X_4[i] - new_c1)
    distances2[i, 1] = np.linalg.norm(X_4[i] - new_c2)

new_assignments = np.argmin(distances2, axis=1)
print(f"New assignments: {new_assignments}")
print(f"Same as before? {np.array_equal(assignments, new_assignments)}")

In [ ]:
# Problem 5: Elbow Method Interpretation
print("=" * 60)
print("PROBLEM 5: Elbow Method Interpretation")
print("=" * 60)

# Given SSE values
k_values = np.array([1, 2, 3, 4, 5, 6])
sse_values = np.array([1200, 650, 420, 350, 330, 320])

print("Given SSE values:")
for k, sse in zip(k_values, sse_values):
    print(f"  K={k}: SSE={sse}")

# Calculate reduction rates
print("\nSSE Reduction Analysis:")
for i in range(1, len(k_values)):
    reduction = sse_values[i-1] - sse_values[i]
    pct_reduction = (reduction / sse_values[i-1]) * 100
    print(f"  K={k_values[i-1]}→{k_values[i]}: reduction={reduction}, %={pct_reduction:.1f}%")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_values, sse_values, 'bo-', linewidth=2, markersize=10)
plt.title('Elbow Method - Problem 5 Data', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('SSE')
plt.grid(True, alpha=0.3)

# Annotate elbow
plt.annotate('Elbow at K=3\n(sharp drop then diminishing returns)', 
             xy=(3, 420), 
             xytext=(4.5, 600),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=12, color='red', fontweight='bold')

plt.show()

print("\nAnswers:")
print("1. Optimal K using Elbow Method: K=3")
print("2. Why K=6 is misleading: SSE always decreases with K; K=n gives SSE=0")
print("3. Sharp reduction K=1→2: Data has strong natural division into 2 clusters")

In [ ]:
# Problem 6: Comparing Runs with Different K
print("=" * 60)
print("PROBLEM 6: Comparing Runs with Different K")
print("=" * 60)

print("Run A: K=2, SSE=250")
print("Run B: K=3, SSE=180")
print()
print("Student concludes: 'Run B is definitely better because its SSE is lower.'")
print()
print("Analysis:")
print("- SSE is NOT directly comparable across different K values")
print("- SSE monotonically decreases as K increases")
print("- Run B has K=3 vs Run A K=2, so lower SSE is expected")
print()
print("Fair comparison requires:")
print("1. Same K value for both runs")
print("2. Elbow method or silhouette score")
print("3. Check if additional cluster captures meaningful structure")
print()
print("Conclusion: Do NOT agree with student. The comparison is invalid.")